# citenode — explore

A thin driver over the public API: it wires the real adapters, runs `verify_claim`, and
renders the result plus the Council's intermediates. No verification logic lives here.

**Prerequisites**
- HelixDB + the TEI embeddings service running locally (see `docker-compose.yml`).
- `.env` (copied from `.env.example`) with `OPENROUTER_API_KEY`, the model picks, and `OPENALEX_MAILTO`.
- Secrets are read from the environment only — never type a key into a cell.

In [ ]:
import os

import httpx
import pyalex
from helixdb import Client
from verdict import config
from verdict.adapters.embeddings import EmbeddingServiceClient
from verdict.adapters.helix_store import HelixGraphVectorStore
from verdict.adapters.openrouter import OpenRouterModelProvider
from verdict.config import Settings
from verdict.council.run import run_council
from verdict.ingest.corpus import ingest_corpus
from verdict.ingest.openalex import OpenAlexSource
from verdict.pipeline import CitenodeDeps, verify_claim
from verdict.retrieval import gather_evidence

## Wire the dependencies

Real adapters, configured from `.env` and `config`. The model picks and key come from `Settings`.

In [ ]:
settings = Settings()  # OPENROUTER_API_KEY + model picks from .env
pyalex.config.email = os.environ.get("OPENALEX_MAILTO")  # OpenAlex polite pool

http_client = httpx.AsyncClient()
embedder = EmbeddingServiceClient(config.EMBEDDINGS_BASE_URL, http_client=http_client)
store = HelixGraphVectorStore(Client(config.HELIX_URL))
await store.ensure_schema()  # create the vector index before any insert

provider = OpenRouterModelProvider(
    api_key=settings.openrouter_api_key,
    cheap_model=settings.openrouter_cheap_model,
    member_models=settings.openrouter_member_models,
    chairman_model=settings.openrouter_chairman_model,
)

deps = CitenodeDeps(
    store=store,
    embedder=embedder,
    text_embedder=embedder,  # one TEI model backs both ports
    source=OpenAlexSource(),
    provider=provider,
    k=12,
    escalation_threshold=0.1,
)

## Ingest a per-claim corpus

Seed-and-expand from OpenAlex into HelixDB. This is bounded (config `INGEST_*`), not a bulk download.

In [ ]:
claim = "Creatine supplementation improves short-term memory."
ingested = await ingest_corpus(claim, source=deps.source, store=store, embedder=embedder)
print(f"ingested {ingested} papers")

## Verify the claim end-to-end

In [ ]:
result = await verify_claim(claim, deps=deps)
print(f"verdict:       {result.verdict}")
print(f"confidence:    {result.confidence.band} ({result.confidence.score:.2f}) — {result.confidence.basis}")
print(f"path:          {result.path}")
print(f"synthesis:     {result.synthesis}")
print(f"dissent:       {result.dissent}")
print(f"supporting:    {[item.paper.openalex_id for item in result.supporting]}")
print(f"contradicting: {[item.paper.openalex_id for item in result.contradicting]}")
print(f"citations:     {[ref.openalex_id for ref in result.citations]}")

## Inspect the evidence and the Council

The value of the notebook: see the retrieved stances and the Council intermediates —
does the dissent surface a consideration the single cheap answer omits?

In [ ]:
evidence = await gather_evidence(claim, store=store, embedder=embedder, model=provider.cheap_model(), k=deps.k)
print(f"balance: {evidence.balance}")
for item in evidence.items:
    print(f"  {item.paper.openalex_id}  {item.stance:<12}  {item.paper.title[:60]}")
print(evidence.coverage_note)

In [ ]:
council = await run_council(claim, evidence, provider=provider, embedder=deps.text_embedder)
for name, member in council.members.items():
    print(f"  {name}: {member.verdict:<11}  +{member.supporting_ids}  -{member.contradicting_ids}")
signals = council.signals
print(
    f"signals: W={signals.kendalls_w}  EU={signals.eu}  disagreement={signals.has_disagreement}  low_info={signals.low_information}"
)
print(f"chairman: {council.chairman.verdict}  confidence={council.confidence.band}")
print(f"dissent:  {council.chairman.dissent}")